In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import geopandas as gpd

In [ ]:
# Paths
SPATIAL_GRIDS_DIR = "../1_data/processed/spatial_grids"
CELL_ZONES_PATH = "../1_data/processed/cell_zones.parquet"
SHAPEFILE_PATH = "../1_data/raw/WarningAreas/WarningAreas.shp"
SAVE_DIR = "../1_data/processed/burned_area"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# Load cell-zone mapping and precomputed fire grids
df_cell_zones = pd.read_parquet(CELL_ZONES_PATH)

yearly_files = sorted([f for f in os.listdir(SPATIAL_GRIDS_DIR) if f.startswith("fire_counts_") and not f.endswith("total.npy")])
print("Found yearly grids:", yearly_files)

In [ ]:
# Extract per-cell fire count for each year using Row/Column indexing
for f in yearly_files:
    year = f.split("_")[2].split(".")[0]
    grid = np.load(os.path.join(SPATIAL_GRIDS_DIR, f))
    df_cell_zones[f"fire_{year}"] = grid[df_cell_zones["Row"].values, df_cell_zones["Column"].values]

year_cols = [c for c in df_cell_zones.columns if c.startswith("fire_")]
df_cell_zones["Fire_Count_Total"] = df_cell_zones[year_cols].sum(axis=1)

In [ ]:
# Rank cells by total fire count
df_fire_ranked = df_cell_zones.sort_values("Fire_Count_Total", ascending=False).reset_index(drop=True)
df_fire_ranked

In [ ]:
# Save full fire count per cell
output_path = os.path.join(SAVE_DIR, "fire_counts_per_cell_all_years.parquet")
df_fire_ranked.to_parquet(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Total land cells: {len(df_fire_ranked):,}")
print(f"Cells with no fire: {(df_fire_ranked['Fire_Count_Total'] == 0).sum():,}")
print(f"Cells with fire: {(df_fire_ranked['Fire_Count_Total'] > 0).sum():,}")
print(f"Max fire count: {df_fire_ranked['Fire_Count_Total'].max()}")

In [ ]:
# Load zone shapefile and compute areas
gdf_zones = gpd.read_file(SHAPEFILE_PATH)
gdf_zones = gdf_zones[gdf_zones.geometry.is_valid]
gdf_zones["Zone_ID"] = gdf_zones["id"].astype(int)
gdf_zones["Area_ha"] = (gdf_zones.geometry.area / 10000).round(2)
gdf_zones["Area_km2"] = (gdf_zones.geometry.area / 1000000).round(2)
df_zone_areas = gdf_zones[["Zone_ID", "Area_ha", "Area_km2"]].sort_values("Zone_ID").reset_index(drop=True)


In [ ]:
# Compute burned area statistics per zone
regional_total = sum(
    (df_fire_ranked[df_fire_ranked["Zone_ID"] == z][year_cols] > 0).values.sum()
    for z in sorted(df_fire_ranked["Zone_ID"].unique())
)

rows = []
for zone_id in sorted(df_fire_ranked["Zone_ID"].unique()):
    df_zone = df_fire_ranked[df_fire_ranked["Zone_ID"] == zone_id]
    total = sum((df_zone[col] > 0).sum() for col in year_cols)
    mean = round(total / len(year_cols), 2)

    zone_area_ha = df_zone_areas[df_zone_areas["Zone_ID"] == zone_id]["Area_ha"].values[0]
    zone_area_km2 = df_zone_areas[df_zone_areas["Zone_ID"] == zone_id]["Area_km2"].values[0]

    rows.append({
        "Zone": int(zone_id),
        "Zone_Area_ha": zone_area_ha,
        "Zone_Area_km2": zone_area_km2,
        "Mean_Annual_Burned_Area_ha": mean,
        "Total_Burned_Area_ha": total,
        "Percent_Burned_in_Zone": round(total / zone_area_ha * 100, 2),
        "Percent_Contribution_Region": round(total / regional_total * 100, 2),
    })

df_table2 = pd.DataFrame(rows)

In [ ]:
# Add total row
total_row = pd.DataFrame([{
    "Zone": "Total",
    "Zone_Area_ha": round(df_table2["Zone_Area_ha"].sum(), 2),
    "Zone_Area_km2": round(df_table2["Zone_Area_km2"].sum(), 2),
    "Mean_Annual_Burned_Area_ha": "-",
    "Total_Burned_Area_ha": regional_total,
    "Percent_Burned_in_Zone": "-",
    "Percent_Contribution_Region": 100.00,
}])

df_table2 = pd.concat([df_table2, total_row], ignore_index=True)
df_table2

In [ ]:
# Save Table 2
csv_path = os.path.join(SAVE_DIR, "table2_burned_area_per_zone.csv")
df_table2.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")